**# Task 4: Data Science Tool Mastery Project**

**Tools:** SQL (SQLite), Plotly, scikit-learn, Pandas

**Data:** cleaned USGS earthquake data for India and nearby regions (2000 to 2026) from Task 3

**Goal:** show proficiency with industry-standard data tools by answering earthquake questions with SQL, interactive visualizations and machine learning.**

**Check the tools**

In [8]:
import sys, sqlite3
import pandas as pd
import numpy as np
import plotly
import sklearn

print("Python       :", sys.version.split()[0])
print("Pandas       :", pd.__version__)
print("NumPy        :", np.__version__)
print("Plotly       :", plotly.__version__)
print("scikit-learn :", sklearn.__version__)
print("SQLite       :", sqlite3.sqlite_version)

Python       : 3.13.18
Pandas       : 2.2.3
NumPy        : 2.1.3
Plotly       : 5.24.1
scikit-learn : 1.6.1
SQLite       : 3.45.1


**Load the data**

In [9]:
import numpy as np

# 1. try the data already in memory, then GitHub, then a manual upload
try:
    df
    print("Using the data already loaded")
except NameError:
    df = None
    base = "https://raw.githubusercontent.com/Santoshgutte1522/data-engineer-internship/main/Task3_Earthquake_Analysis/"
    for name in ["india_earthquakes_clean.csv", "india_earthquakes_raw.csv"]:
        try:
            df = pd.read_csv(base + name)
            print("Loaded from GitHub:", name)
            break
        except Exception as e:
            print("Could not load", name, "->", e)
    if df is None:
        from google.colab import files
        uploaded = files.upload()
        df = pd.read_csv(list(uploaded.keys())[0])

# 2. cleaning (same steps as Task 3, safe to run on the raw or the clean file)
if "type" in df.columns:
    df = df[df["type"] == "earthquake"].drop(columns=["type"])   # keep only earthquakes
df = df.drop_duplicates(subset="id").copy()

df["time"] = pd.to_datetime(df["time"], utc=True)
df["year"] = df["time"].dt.year
df["month"] = df["time"].dt.month
df["time"] = df["time"].dt.strftime("%Y-%m-%d %H:%M:%S")   # text, so SQL can read it

# 3. country names
if "country" not in df.columns:
    df["country"] = df["place"].str.split(",").str[-1].str.strip()

country_fix = {
    "Burma (Myanmar)": "Myanmar",
    "India region": "India",
    "western Xizang": "China",
    "Southern Tibetan Plateau": "China",
    "Xizang-Qinghai border region": "China",
}
df["country"] = df["country"].replace(country_fix)
counts = df["country"].value_counts()
small = counts[counts < 50].index
df["country_group"] = np.where(df["country"].isin(small), "Other", df["country"])

print("Shape:", df.shape)
print(df[["time", "latitude", "longitude", "depth", "mag", "year", "country_group"]].head())
print()
print(df["country_group"].value_counts())

Using the data already loaded
Shape: (15381, 25)
                  time  latitude  longitude  depth  mag  year country_group
0  2000-01-01 05:24:35    36.874     69.947   54.3  5.1  2000   Afghanistan
1  2000-01-02 10:21:17    36.229     70.893   33.0  4.1  2000   Afghanistan
2  2000-01-02 10:23:58    27.559     92.498   33.0  5.0  2000         India
3  2000-01-03 22:34:12    22.132     92.771   33.0  4.6  2000         India
4  2000-01-05 09:45:18    32.222     92.700   33.0  5.5  2000         China

country_group
India          5073
Afghanistan    3658
China          1754
Indonesia      1551
Myanmar        1247
Pakistan        999
Nepal           773
Other           171
Bangladesh       78
Tajikistan       77
Name: count, dtype: int64


**Put the data into a SQL database**

In [10]:
# create a small database in memory and load the table into it
conn = sqlite3.connect(":memory:")
df.to_sql("earthquakes", conn, index=False, if_exists="replace")

# helper: run a SQL query and show the result as a table
def run(sql):
    return pd.read_sql_query(sql, conn)

# see the table structure
print(run("PRAGMA table_info(earthquakes)")[["name", "type"]])
print()
print(run("SELECT COUNT(*) AS total_rows FROM earthquakes"))

               name     type
0              time     TEXT
1          latitude     REAL
2         longitude     REAL
3             depth     REAL
4               mag     REAL
5           magType     TEXT
6               nst     REAL
7               gap     REAL
8              dmin     REAL
9               rms     REAL
10              net     TEXT
11               id     TEXT
12          updated     TEXT
13            place     TEXT
14  horizontalError     REAL
15       depthError     REAL
16         magError     REAL
17           magNst     REAL
18           status     TEXT
19   locationSource     TEXT
20        magSource     TEXT
21             year  INTEGER
22            month  INTEGER
23          country     TEXT
24    country_group     TEXT

   total_rows
0       15381


In [11]:
# fix records whose place text is an event name, e.g. "2025 Mandalay, Burma (Myanmar) Earthquake"
def fix_country(text):
    text = text.replace(" Earthquake", "").strip()   # remove the word Earthquake
    text = text.lstrip("0123456789 ")                # remove a leading year
    return country_fix.get(text, text)

df["country"] = df["country"].apply(fix_country)
counts = df["country"].value_counts()
small = counts[counts < 50].index
df["country_group"] = np.where(df["country"].isin(small), "Other", df["country"])
print(df["country_group"].value_counts())

country_group
India          5073
Afghanistan    3658
China          1755
Indonesia      1551
Myanmar        1248
Pakistan        999
Nepal           773
Other           169
Bangladesh       78
Tajikistan       77
Name: count, dtype: int64


**SQL exercises 1 to 4**

In [12]:
# 1. SELECT with ORDER BY and LIMIT: top 5 strongest earthquakes
print("1. Top 5 strongest earthquakes")
print(run("""
    SELECT substr(time, 1, 10) AS date, mag, depth, place
    FROM earthquakes
    ORDER BY mag DESC
    LIMIT 5
"""))

# 2. GROUP BY: summary for each country
print("\n2. Summary by country")
print(run("""
    SELECT country_group,
           COUNT(*)            AS quakes,
           ROUND(AVG(mag), 2)  AS avg_mag,
           MAX(mag)            AS max_mag,
           ROUND(AVG(depth),1) AS avg_depth
    FROM earthquakes
    GROUP BY country_group
    ORDER BY quakes DESC
"""))

# 3. HAVING: years with more than 600 earthquakes
print("\n3. Years with more than 600 earthquakes")
print(run("""
    SELECT year, COUNT(*) AS quakes
    FROM earthquakes
    GROUP BY year
    HAVING COUNT(*) > 600
    ORDER BY quakes DESC
"""))

# 4. CASE WHEN: group earthquakes by strength
print("\n4. Earthquakes by strength class")
print(run("""
    SELECT CASE
             WHEN mag >= 7 THEN 'Major (7+)'
             WHEN mag >= 6 THEN 'Strong (6-6.9)'
             WHEN mag >= 5 THEN 'Moderate (5-5.9)'
             ELSE 'Light (4-4.9)'
           END AS strength,
           COUNT(*) AS quakes
    FROM earthquakes
    GROUP BY strength
    ORDER BY MIN(mag) DESC
"""))

1. Top 5 strongest earthquakes
         date  mag  depth                                      place
0  2001-11-14  7.8  10.00                    Southern Qinghai, China
1  2015-04-25  7.8   8.22              67 km NNE of Bharatpur, Nepal
2  2001-01-26  7.7  16.00                 17 km NW of Bhach?u, India
3  2025-03-28  7.7  10.00  2025 Mandalay, Burma (Myanmar) Earthquake
4  2005-10-08  7.6  26.00        21 km NNE of Muzaffar?b?d, Pakistan

2. Summary by country
  country_group  quakes  avg_mag  max_mag  avg_depth
0         India    5073     4.54      7.7       32.9
1   Afghanistan    3658     4.37      7.5      143.4
2         China    1754     4.51      7.8       19.5
3     Indonesia    1551     4.61      7.2       32.4
4       Myanmar    1247     4.50      6.9       48.7
5      Pakistan     999     4.49      7.6       22.9
6         Nepal     773     4.47      7.8       16.1
7         Other     171     4.50      7.7       49.5
8    Bangladesh      78     4.44      5.7       21.2
9 

**SQL exercises 5 to 8**

In [13]:
# 5. SUBQUERY: India earthquakes stronger than the overall average
print("5. India earthquakes stronger than the overall average magnitude")
print(run("""
    SELECT COUNT(*) AS stronger_than_average
    FROM earthquakes
    WHERE country_group = 'India'
      AND mag > (SELECT AVG(mag) FROM earthquakes)
"""))

# 6. CTE + WINDOW FUNCTION: top 3 strongest quakes in each country
print("\n6. Top 3 strongest earthquakes in each country")
print(run("""
    WITH ranked AS (
        SELECT country_group, substr(time, 1, 10) AS date, mag, place,
               RANK() OVER (PARTITION BY country_group ORDER BY mag DESC) AS rnk
        FROM earthquakes
    )
    SELECT country_group, date, mag, place
    FROM ranked
    WHERE rnk <= 3
    ORDER BY country_group, rnk
"""))

# 7. WINDOW FUNCTION: running total of earthquakes over the years
print("\n7. Running total by year (first 8 years)")
print(run("""
    SELECT year,
           COUNT(*) AS quakes,
           SUM(COUNT(*)) OVER (ORDER BY year) AS running_total
    FROM earthquakes
    GROUP BY year
    ORDER BY year
    LIMIT 8
"""))

# 8. LAG: change from the previous year
print("\n8. Change from the previous year (top 5 jumps)")
print(run("""
    WITH yearly AS (
        SELECT year, COUNT(*) AS quakes
        FROM earthquakes
        GROUP BY year
    )
    SELECT year, quakes,
           quakes - LAG(quakes) OVER (ORDER BY year) AS change_from_last_year
    FROM yearly
    ORDER BY change_from_last_year DESC
    LIMIT 5
"""))

5. India earthquakes stronger than the overall average magnitude
   stronger_than_average
0                   2664

6. Top 3 strongest earthquakes in each country
   country_group        date  mag                                      place
0    Afghanistan  2015-10-26  7.5             Hindu Kush region, Afghanistan
1    Afghanistan  2002-03-03  7.4              51 km SW of Jurm, Afghanistan
2    Afghanistan  2009-01-03  6.6               50 km S of Jurm, Afghanistan
3    Afghanistan  2016-04-10  6.6        42 km WSW of Ashk?sham, Afghanistan
4     Bangladesh  2003-07-26  5.7       44 km SE of Khagrachhari, Bangladesh
5     Bangladesh  2007-11-07  5.5         18 km ESE of B?ndarban, Bangladesh
6     Bangladesh  2023-12-02  5.5            9 km ENE of Rāmganj, Bangladesh
7          China  2001-11-14  7.8                    Southern Qinghai, China
8          China  2008-03-20  7.2                  226 km SE of Hotan, China
9          China  2026-10-22  7.0                 191 km ENE of Nag

**Prepare**

In [14]:
import plotly.express as px

# strength class: 6.0 counts as Strong (right=False), so 5.95 stays Moderate
df["strength"] = pd.cut(df["mag"], bins=[4, 5, 6, 7, 10], right=False,
                        labels=["Light (4-4.9)", "Moderate (5-5.9)", "Strong (6-6.9)", "Major (7+)"])
df["date"] = df["time"].str[:10]
df["marker_size"] = (df["mag"] - 3) ** 2     # bigger earthquake = bigger dot

print(df["strength"].value_counts())

strength
Light (4-4.9)       13631
Moderate (5-5.9)     1635
Strong (6-6.9)        100
Major (7+)             15
Name: count, dtype: int64


**Chart 1, interactive map of strong earthquakes**

In [15]:
big = df[df["mag"] >= 5]     # magnitude 5 and above, so the map stays readable

fig = px.scatter_geo(
    big, lat="latitude", lon="longitude",
    color="mag", size="marker_size",
    hover_name="place",
    hover_data={"date": True, "depth": True, "mag": True,
                "latitude": False, "longitude": False, "marker_size": False},
    color_continuous_scale="YlOrRd",
    projection="natural earth",
    title="Earthquakes of magnitude 5 and above (2000 to 2026)")
fig.update_geos(lonaxis_range=[60, 105], lataxis_range=[0, 42],
                showcountries=True, countrycolor="gray",
                showland=True, landcolor="whitesmoke")
fig.update_layout(height=600)
fig.show()

**Charts 2 and 3**

In [16]:
# Chart 2: earthquakes per year, split by country (shows what built the 2005 spike)
yearly = df.groupby(["year", "country_group"]).size().reset_index(name="quakes")
fig = px.bar(yearly, x="year", y="quakes", color="country_group",
             title="Earthquakes per Year by Country / Region",
             labels={"quakes": "Number of earthquakes", "country_group": "Country / region"})
fig.update_layout(height=450)
fig.show()

# Chart 3: sunburst, click a country to see its strength classes
fig = px.sunburst(df, path=["country_group", "strength"],
                  title="Earthquakes by Country and Strength (click to zoom)")
fig.update_layout(height=550)
fig.show()

/usr/local/lib/python3.13/dist-packages/plotly/express/_core.py:1727: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



**Charts 4 and 5**

In [17]:
# Chart 4: depth by country (depth is drawn downwards, like real depth)
top = df["country_group"].value_counts().head(7).index
fig = px.box(df[df["country_group"].isin(top)], x="country_group", y="depth",
             color="country_group", points=False,
             title="Earthquake Depth by Country / Region",
             labels={"depth": "Depth (km)", "country_group": "Country / region"})
fig.update_yaxes(autorange="reversed")
fig.update_layout(height=450, showlegend=False)
fig.show()

# Chart 5: density heatmap, where are the hotspots?
fig = px.density_heatmap(df, x="longitude", y="latitude", nbinsx=60, nbinsy=60,
                         color_continuous_scale="YlOrRd",
                         title="Earthquake Hotspots (number of earthquakes per grid square)")
fig.update_layout(height=550)
fig.show()

**Chart 6, animated map by year**

In [18]:
# press the play button to watch the earthquakes year by year
anim = df[df["mag"] >= 5].sort_values("year")

fig = px.scatter_geo(
    anim, lat="latitude", lon="longitude",
    color="mag", size="marker_size", animation_frame="year",
    hover_name="place", range_color=[5, 8],
    color_continuous_scale="YlOrRd", projection="natural earth",
    title="Earthquakes of magnitude 5 and above, year by year")
fig.update_geos(lonaxis_range=[60, 105], lataxis_range=[0, 42],
                showcountries=True, countrycolor="gray",
                showland=True, landcolor="whitesmoke")
fig.update_layout(height=600)
fig.show()

**Group earthquakes into zones (KMeans clustering)**

In [19]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

coords = df[["latitude", "longitude"]]

# try different numbers of zones (k) and compare the scores
print("k   inertia   silhouette")
for k in range(3, 10):
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(coords)
    score = silhouette_score(coords, labels, sample_size=3000, random_state=42)
    print(k, round(model.inertia_), round(score, 3))

k   inertia   silhouette
3 336359 0.69
4 215691 0.669
5 172025 0.66
6 134967 0.659
7 111968 0.587
8 88665 0.587
9 78968 0.561


**Build 6 zones and show them on a map**

In [20]:
# k = 6 zones (a good balance between detail and readability)
kmeans = KMeans(n_clusters=6, random_state=42, n_init=10)
labels = kmeans.fit_predict(coords)
df["zone"] = ["Zone " + str(z + 1) for z in labels]

# summary of each zone
zone_summary = df.groupby("zone").agg(
    quakes=("mag", "size"),
    center_lat=("latitude", "mean"),
    center_lon=("longitude", "mean"),
    avg_mag=("mag", "mean"),
    avg_depth=("depth", "mean"),
    top_country=("country_group", lambda s: s.value_counts().index[0]),
).round(2)
print(zone_summary)

# map of the zones (5000 random points so the chart stays light)
sample = df.sample(5000, random_state=42)
fig = px.scatter_geo(sample, lat="latitude", lon="longitude", color="zone",
                     hover_name="place", hover_data={"mag": True, "depth": True},
                     category_orders={"zone": sorted(df["zone"].unique())},
                     projection="natural earth",
                     title="Seismic Zones Found by KMeans Clustering")
fig.update_geos(lonaxis_range=[60, 105], lataxis_range=[0, 42],
                showcountries=True, countrycolor="gray",
                showland=True, landcolor="whitesmoke")
fig.update_layout(height=600)
fig.show()

        quakes  center_lat  center_lon  avg_mag  avg_depth  top_country
zone                                                                   
Zone 1    1017       32.39       93.22     4.55      19.56        China
Zone 2    4977        9.76       93.44     4.58      30.48        India
Zone 3    4938       35.90       71.56     4.40     115.30  Afghanistan
Zone 4    2067       23.42       94.43     4.49      52.08      Myanmar
Zone 5    1947       30.92       84.11     4.48      17.94        China
Zone 6     435       25.73       70.84     4.46      14.72        India


**Predict deep or shallow, comparing 3 models**

In [21]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score

# target: 1 = deep earthquake (70 km or more), 0 = shallow
df["is_deep"] = (df["depth"] >= 70).astype(int)
features = ["latitude", "longitude", "mag"]
X = df[features]
y = df["is_deep"]
print("Share of deep earthquakes:", round(y.mean() * 100, 1), "%")

# split: 80% to train the model, 20% to test it
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

models = {
    "Baseline (always predict shallow)": DummyClassifier(strategy="most_frequent"),
    "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
}

rows = []
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    rows.append([name,
                 round(accuracy_score(y_test, pred), 3),
                 round(f1_score(y_test, pred, zero_division=0), 3)])

print(pd.DataFrame(rows, columns=["Model", "Accuracy", "F1 (deep class)"]))

Share of deep earthquakes: 27.4 %
                               Model  Accuracy  F1 (deep class)
0  Baseline (always predict shallow)     0.726            0.000
1                Logistic Regression     0.839            0.711
2                      Random Forest     0.945            0.898


**Check the best model properly**

In [22]:
from sklearn.metrics import confusion_matrix, classification_report

rf = models["Random Forest"]
pred = rf.predict(X_test)

print(classification_report(y_test, pred, target_names=["Shallow", "Deep"]))

cm = pd.DataFrame(confusion_matrix(y_test, pred),
                  index=["Actual shallow", "Actual deep"],
                  columns=["Predicted shallow", "Predicted deep"])
print(cm)

# which feature matters most?
importance = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False)
print("\nFeature importance:")
print(importance.round(3))

# a harder test: train on 2000-2018, test on 2019-2026 (data the model has never seen)
train_part = df[df["year"] <= 2018]
test_part = df[df["year"] >= 2019]
rf2 = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf2.fit(train_part[features], train_part["is_deep"])
pred2 = rf2.predict(test_part[features])
baseline = 1 - test_part["is_deep"].mean()
print("\nTime-based test (2019-2026):")
print("Random Forest accuracy:", round(accuracy_score(test_part["is_deep"], pred2), 3))
print("Baseline accuracy     :", round(baseline, 3))

# charts
fig = px.imshow(cm, text_auto=True, color_continuous_scale="Blues",
                title="Confusion Matrix: Random Forest")
fig.show()

fig = px.bar(x=importance.index, y=importance.values,
             labels={"x": "Feature", "y": "Importance"},
             title="Which Feature Predicts Depth Best?")
fig.show()

              precision    recall  f1-score   support

     Shallow       0.95      0.97      0.96      2233
        Deep       0.92      0.88      0.90       844

    accuracy                           0.95      3077
   macro avg       0.94      0.92      0.93      3077
weighted avg       0.94      0.95      0.94      3077

                Predicted shallow  Predicted deep
Actual shallow               2169              64
Actual deep                   104             740

Feature importance:
latitude     0.496
longitude    0.473
mag          0.031
dtype: float64

Time-based test (2019-2026):
Random Forest accuracy: 0.942
Baseline accuracy     : 0.665
